In [1]:
import sys
#sys.path.insert(1, '/Feature_extraction')    #some important scripts in here, visualizations, custom datasets, etc
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision as tv
import numpy as np
import glob
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from torchvision.utils import save_image
import torchvision
import torch
from IPython.display import display,clear_output
#import backbone.VISUAL as viz
import backbone.Imagefolder as imf
import importlib
import pickle
import backbone.Custom as cust
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split



# Define the directory of your dataset
meerkat_dir = "/idia/projects/hippo/Koketso/meerkat"
dogbreed_dir = "/idia/projects/hippo/Koketso/dog_breeds"
galaxyzooc_dir = "/idia/projects/hippo/Koketso/galaxyzoo/galaxy_zoo_biclass_clipped_resized"
hand_dir = "/idia/projects/hippo/Koketso/Train_Alphabet"
handt_dir = "/idia/projects/hippo/Koketso/Test_Alphabet"
im13_dir = "/idia/projects/hippo/Koketso/ILSVRC/Data/DET/train/ILSVRC2013_train"
im12_test =  "/idia/projects/hippo/Koketso/ILSVRC/Data/DET/test"

/idia/projects/hippo/Koketso/deepclustering2/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
#Define and load model

device = torch.device('cuda') if torch.cuda.is_available() else torch.device("cpu")
torch.cuda.empty_cache()

model = torchvision.models.efficientnet_b0(weights = "IMAGENET1K_V1")
#load the zoobot weights(efficientnetb0)

zoobot_weights = torch.load("models/checkpoints/epoch=17-step=2808.ckpt")['state_dict']
#adjust for mismatching keys

loaded_dict = {k: zoobot_weights["model.0.features." +k] for k in model.features.state_dict()}
model.features.load_state_dict(loaded_dict)

#The classifier

model.classifier[1] = torch.nn.Linear(in_features=1280, out_features=500, bias=True)
model.classifier.append(torch.nn.Dropout(p=0.1, inplace=True))
model.classifier.append(torch.nn.Linear(in_features = 500,out_features = 2))
model.classifier[1].weight.data.normal_(0,0.01)
model.classifier[3].weight.data.normal_(0,0.01)

model.features.train(mode = False)
model.classifier.train()
model.to(device)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [14]:
# Load data
Dir = galaxyzooc_dir
images = torchvision.datasets.ImageFolder(Dir)
labels = imf.classes(Dir)
print(max(labels))
pytorch_dataset = cust.Custom_labelled(images,labels)

1


In [16]:
def train_test_dataset(dataset, val_split=0.1, test_split = 0.1):
    train_idx, val_idx = train_test_split(list(range(len(dataset))), test_size=val_split)
    datasets = {}
    datasets['train'] = Subset(dataset, train_idx)
    datasets['val'] = Subset(dataset, val_idx)
    datasets['test'] = Subset(dataset, test_split)
    return datasets    


data = train_test_dataset(pytorch_dataset)
data

{'train': <torch.utils.data.dataset.Subset at 0x7ff17d2621f0>,
 'val': <torch.utils.data.dataset.Subset at 0x7ff17d262310>,
 'test': <torch.utils.data.dataset.Subset at 0x7ff17d2620a0>}

In [17]:
batch_size = 25
val_batch_size = 25
epochs = 500

trainloader = DataLoader(data['train'], 
                    batch_size, 
                    shuffle = True )

val_loader = DataLoader(data['val'], 
                    val_batch_size, 
                    shuffle = True )


criterion = torch.torch.nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
            model.classifier.parameters(),
            lr=0.15)
            #momentum=args.momentum,
            #weight_decay=10**
        

In [ ]:
train_losses = []
val_losses = []
c = 10
train_loss_saved = "traingzzoobottloss.plk"
val_loss_saved = "testgzzoobotvloss.plk"
for epoch in range(epochs):
    train_running_loss = 0.0
    val_loss = 0.0
    correct = 0
    model.classifier.train()
    for i, batch in enumerate(trainloader):
    
        inputs,labels = batch
        inputs = inputs.to(device)
        labels = labels.to(device)
    
        
        #zero the gradients
        optimizer.zero_grad()
        
        #forwad + backward + optimize
        
        outputs = model(inputs)
        loss = criterion(outputs,labels)
        
        #computing the accuracy 
        
        loss.backward()
        optimizer.step()
        
        train_running_loss += loss.item()
        #print progress every once in a while
        #model.classifier.train(True)
        if i%c==0:
            print("Epoch ", epoch," Batch", i)

    print("Validating")
    model.classifier.eval()
    for j,batch in enumerate(val_loader):
                
        vinputs,vlabels = batch
        vinputs = vinputs.to(device)
        vlabels = vlabels.to(device)
        voutputs = model(vinputs)
                
        val_loss  += criterion(voutputs,vlabels).item()
        #match the train and val losses
        
    val_losses.append(val_loss)
    train_losses.append(train_running_loss)
    clear_output()

    print('[%d,%5d] Train loss: %.3f' %(epoch + 1, i + 1,train_running_loss),'[%d,%5d] Validation loss: %.3f' %(epoch + 1, i + 1,val_loss ))
          

    train_running_loss =0.0
    val_loss = 0
    correct = 0
 

   
    # update the loss array and model state dictionary every epoch 
    with open(train_loss_saved,'wb') as file:
        pickle.dump(train_losses,file)                                            #
    with open(val_loss_saved,'wb') as file:
        pickle.dump(val_losses,file)                                            #
    torch.save(model.state_dict(),"zoobotbyclass.pth")
print("Done")

[13,  360] Train loss: 143.800 [13,  360] Validation loss: 15.790
Epoch  13  Batch 0
Epoch  13  Batch 10
Epoch  13  Batch 20
Epoch  13  Batch 30
Epoch  13  Batch 40
Epoch  13  Batch 50
Epoch  13  Batch 60


In [ ]:


batch_size = 50
validation_loader = DataLoader(data['test'], 
                    batch_size, 
                    shuffle = True )
correct =0
total = 0
for i, vdata in enumerate(validation_loader):
    vinputs, vlabels = vdata
    
    #send data to device
    vinputs = vinputs.to(device)
    vlabels = vlabels.to(device)
    
    #compute the output
    outputs = model(vinputs)
    
    #loss of for comparision
    loss = criterion(outputs, vlabels)
    
    #compute the correctly classified using this method I wont remember (compare matching predictions)
    correct += sum([outputs.data.topk(1, dim=-1).indices[a] == vlabels[a] for a in range(len(vlabels))])
    total+=batch_size
    print(correct.item()*100/total,loss.item())